# Prompt Optimization

Source: https://arxiv.org/pdf/2507.19457

## What is Prompt Optimization?

Prompt optimization is the process of systematically improving your AI prompts to achieve better performance, accuracy, and consistency. Instead of manually tweaking prompts through trial and error, MLflow uses **GEPA (Generated Prompt Adaptation)** - an LLM-driven algorithm that automatically refines prompts using your evaluation data.

### Key Benefits:
- **Minimal Code Changes**: Add just a few lines to start optimizing
- **Framework Agnostic**: Works with LangChain, OpenAI Agent, CrewAI, DSPy, or custom implementations
- **Zero Lock-in**: No vendor dependency; seamless integration with your existing setup
- **Data-Driven**: Learn from real examples and custom metrics
- **Multi-Prompt Support**: Optimize single or multiple prompts in complex workflows
- **Production-Ready**: Built-in version control and registry for deployment

### When to Use Prompt Optimization:
✅ Your prompts produce correct but unclear outputs  
✅ Accuracy is below expectations  
✅ You need consistency across different inputs  
✅ Migrating to a new/cheaper model  
✅ You have ground-truth evaluation data  

### Requirements:
- MLflow >= 3.5.0
- Training dataset with inputs and expected outputs
- An LLM API (OpenAI, Azure, etc.)
- Evaluation metrics (scorers)

In [2]:
# Install required packages (uncomment if needed)
# !pip install mlflow>=3.5.0 openai

import mlflow
import openai
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.scorers import Correctness
import pandas as pd
from typing import Any, Dict

print(f"MLflow version: {mlflow.__version__}")

C:\Users\Anshu Pandey\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


MLflow version: 3.6.0


In [18]:
mlflow.set_tracking_uri("http://20.75.92.162:5000/")

## Step 1: Register Your Initial Prompt

The first step is to register your baseline prompt in the MLflow Prompt Registry. This version will serve as the starting point for optimization.

### What is a Prompt Template?
A prompt template uses **double curly braces `{{ }}` syntax** to define placeholders for dynamic variables that will be filled at runtime.

**Example**: The variable `{{sentence}}` will be replaced with actual sentence data when the prompt is executed.

### Registration Process:
You provide:
1. **name**: Unique identifier for your prompt
2. **template**: The actual prompt text with variables
3. This creates version 1 (v1) in the registry

In [19]:
# === Example 1: Medical Paper Section Classification ===
# Register the initial prompt
prompt = mlflow.genai.register_prompt(
    name="medical_section_classifier",
    template="Classify this medical research paper sentence into one of these sections: CONCLUSIONS, RESULTS, METHODS, OBJECTIVE, BACKGROUND.\n\nSentence: {{sentence}}"
)

print(f"Initial prompt registered with URI: {prompt.uri}")
print(f"Template:\n{prompt.template}")

2025/11/21 16:47:47 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: medical_section_classifier, version 1


Initial prompt registered with URI: prompts:/medical_section_classifier/1
Template:
Classify this medical research paper sentence into one of these sections: CONCLUSIONS, RESULTS, METHODS, OBJECTIVE, BACKGROUND.

Sentence: {{sentence}}


## Step 2: Define Your Prediction Function

The prediction function is the heart of optimization. It:
1. **Loads** the prompt template from MLflow registry
2. **Formats** it with actual input data
3. **Calls** your LLM (OpenAI, Azure, etc.)
4. **Returns** the output in the expected format

### Key Requirements:
- ✅ Accept inputs as **keyword arguments** matching your dataset columns
- ✅ Load template using `mlflow.genai.load_prompt()`
- ✅ Format template using `prompt.format()` or `prompt.to_single_brace_format()`
- ✅ Return outputs in **same format as expectations** (dict, string, etc.)

### Template Format Methods:
| Method | Use Case |
|--------|----------|
| `prompt.format()` | Standalone MLflow usage (double braces) |
| `prompt.to_single_brace_format()` | LangChain compatibility (single braces) |
| `prompt.template` | Raw template access |

In [20]:
from openai import AzureOpenAI
client = AzureOpenAI()

In [21]:
# Step 2: Define prediction function
def predict_fn(sentence: str) -> str:
    """
    Prediction function that:
    1. Loads the prompt from MLflow registry
    2. Formats it with the input sentence
    3. Calls OpenAI API
    4. Returns the classification result
    """
    # Load the latest version of the prompt from registry
    loaded_prompt = mlflow.genai.load_prompt("prompts:/medical_section_classifier@latest")
    
    # Format the prompt template with the actual sentence
    formatted_prompt = loaded_prompt.format(sentence=sentence)
    
    # Call OpenAI (replace with your LLM API)
    # Note: In production, use your actual API key and error handling
    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",  # Use your preferred model
            messages=[{"role": "user", "content": formatted_prompt}],
            temperature=0.3,  # Lower temperature for classification tasks
            max_tokens=20
        )
        result = completion.choices[0].message.content.strip()
        return result
    except Exception as e:
        print(f"Note: LLM call would be made here. Error: {e}")
        return "RESULTS"  # Placeholder for demo

print("Prediction function defined successfully")

Prediction function defined successfully


## Step 3: Prepare Training Data

The training dataset is crucial for optimization. It consists of:
- **inputs**: Dictionary with keys matching your prediction function parameters
- **expectations**: Dictionary with expected outputs for evaluation

### Data Format:
```python
dataset = [
    {
        "inputs": {"sentence": "Your input text..."},
        "expectations": {"expected_response": "EXPECTED_LABEL"}
    },
    ...
]
```

### Best Practices for Training Data:
1. **Representative**: Include diverse examples covering all categories/cases
2. **Correct**: All expected_response values must be ground truth
3. **Sufficient**: At least 20-30 examples (more is better for complex tasks)
4. **Balanced**: If possible, balance examples across classes
5. **Real-world**: Use actual production data when available

### Common Dataset Issues to Avoid:
❌ Duplicate or near-duplicate examples  
❌ Incorrect ground truth labels  
❌ Too few examples (< 10)  
❌ All examples from same category  
❌ Format mismatch between inputs and expectations

In [22]:
# Step 3: Prepare training dataset
# Medical paper sentences with correct classifications
raw_data = [
    ("The emergence of HIV as a chronic condition means that people living with HIV are required to take more responsibility for the self-management of their condition.", "BACKGROUND"),
    ("This paper describes the design and evaluation of an online program aiming to enhance self-management skills.", "BACKGROUND"),
    ("This study is designed as a randomised controlled trial where participants will be assigned to intervention or control groups.", "METHODS"),
    ("The intervention group will participate in the online program over seven weeks.", "METHODS"),
    ("The program is based on self-efficacy theory and uses a self-management approach to enhance skills and confidence.", "METHODS"),
    ("Primary outcomes are domain specific self-efficacy, HIV related quality of life, and health education outcomes.", "METHODS"),
    ("Data collection will take place at baseline, completion, and 12-week follow-up.", "METHODS"),
    ("Both groups showed improvement in symptoms and clinical evidence of inflammation.", "RESULTS"),
    ("Mean exophthalmometry values decreased from 22.6 mm to 18.6 mm in group 1, and from 23 mm to 19.08 mm in group 2.", "RESULTS"),
    ("There was no change in best-corrected visual acuity in both groups.", "RESULTS"),
    ("Results of this study will provide information regarding the effectiveness of online programs.", "CONCLUSIONS"),
    ("Orbital steroid injection for thyroid-related ophthalmopathy is effective and safe.", "CONCLUSIONS"),
    ("It eliminates the adverse reactions associated with oral corticosteroid use.", "CONCLUSIONS"),
    ("The aim of this study was to evaluate the efficacy and safety of orbital steroid injection.", "OBJECTIVE"),
    ("We sought to examine whether counseling and oral fluid intake decrease postoperative complications.", "OBJECTIVE"),
]

# Convert to MLflow-compatible format
dataset = [
    {
        "inputs": {"sentence": sentence},
        "expectations": {"expected_response": label}
    }
    for sentence, label in raw_data
]

print(f"Created dataset with {len(dataset)} examples")
print(f"\nFirst example:")
print(f"  Input: {dataset[0]['inputs']['sentence'][:80]}...")
print(f"  Expected: {dataset[0]['expectations']['expected_response']}")

Created dataset with 15 examples

First example:
  Input: The emergence of HIV as a chronic condition means that people living with HIV ar...
  Expected: BACKGROUND


## Step 4: Configure the Optimizer

MLflow provides **GepaPromptOptimizer**, which uses the GEPA algorithm to:
1. Run predictions on your training data
2. Evaluate outputs against expected results
3. Reflect on failures and reasons
4. Generate improved prompt versions
5. Iteratively refine until convergence

### Optimizer Parameters:
| Parameter | Description | Default |
|-----------|-------------|---------|
| `reflection_model` | Model used for LLM-driven reflection (should be powerful) | Required |
| `max_metric_calls` | Maximum LLM API calls budget | 100-1000 |
| `display_progress_bar` | Show optimization progress | False |

### Model Selection:
- **Reflection Model** (✅ should be powerful): `gpt-4`, `gpt-4-turbo` - used for thinking about improvements
- **Scorer Model** (✅ can be cheaper): `gpt-3.5-turbo`, `gpt-4-mini` - used for evaluation
- **Production Model** (in predict_fn): Can be any model for your use case

### Why Use Different Models?
💰 Cost optimization - powerful models for thinking, cheaper for scoring  
⚡ Performance - use best model where quality matters most  
🎯 Flexibility - adapt to your budget constraints

In [23]:
# Step 4: Create and configure the optimizer
optimizer = GepaPromptOptimizer(
    reflection_model="openai:/gpt-4.1-mini",  # Use gpt-4 for best reflection/thinking
    max_metric_calls=300,               # Allow up to 300 LLM calls for optimization
    display_progress_bar=True           # Show progress during optimization
)

print("Optimizer configured:")
print(f"  Reflection Model: gpt-4")
print(f"  Max Metric Calls: 300")
print(f"  This controls the optimization budget and quality vs. cost tradeoff")

Optimizer configured:
  Reflection Model: gpt-4
  Max Metric Calls: 300
  This controls the optimization budget and quality vs. cost tradeoff


## Step 5: Define Evaluation Scorers

Scorers measure how well your prompts perform. MLflow provides built-in scorers and supports custom ones.

### Built-in Scorers:
| Scorer | Purpose | Use For |
|--------|---------|---------|
| `Correctness` | Exact match or semantic similarity | Classification, Q&A |
| `Safety` | Check output safety | Any task (content moderation) |
| `Relevance` | Check output relevance to input | Search, retrieval |
| `Fluency` | Check output readability | Text generation |

### Custom Scorer Example:
```python
@scorer
def custom_scorer(outputs: Any, expectations: dict) -> float:
    # Your custom logic to score outputs (0.0 to 1.0)
    return 1.0 if condition else 0.0
```

### How Scorers Are Used in Optimization:
1. **Initial Evaluation**: Score baseline prompt on all training examples
2. **Candidate Testing**: Score each optimized prompt version
3. **Guidance**: Optimization algorithm uses scores to guide improvements
4. **Comparison**: Compare initial vs. final performance

In [24]:
# Step 5: Define evaluation scorers

# Use built-in Correctness scorer for this classification task
scorers = [
    Correctness(model="openai:/gpt-4o-mini")  # Cheaper model for scoring is fine
]

# Example: Define a custom scorer for additional metrics (optional)
from mlflow.genai.scorers import scorer

@scorer
def classification_confidence_scorer(outputs: Any, expectations: dict) -> float:
    """
    Custom scorer that checks if output is one of valid categories.
    This helps guide optimization towards valid outputs.
    """
    valid_categories = {"CONCLUSIONS", "RESULTS", "METHODS", "OBJECTIVE", "BACKGROUND"}
    output_str = str(outputs).strip().upper()
    
    # Return 1.0 if valid category, 0.5 if partially valid, 0.0 if invalid
    if output_str in valid_categories:
        return 1.0
    elif any(cat in output_str for cat in valid_categories):
        return 0.5
    else:
        return 0.0

print("Scorers configured:")
print("  - Correctness: Checks if output matches expected classification")
print("  - Custom scorer: Validates output is from allowed categories")

Scorers configured:
  - Correctness: Checks if output matches expected classification
  - Custom scorer: Validates output is from allowed categories


## Step 6: Run the Optimization

This is where the magic happens! MLflow will:
1. 🔍 **Evaluate** your initial prompt on all training examples
2. 📊 **Analyze** failures and understand why predictions are incorrect
3. 💡 **Generate** improved prompt versions using LLM reflection
4. 🔄 **Iterate** through multiple refinement cycles
5. 📈 **Track** performance improvement across iterations

### What Happens During Optimization:
```
Initial Prompt (score: 0.60)
    ↓
GEPA Iteration 1: Analyze failures, generate improvement
    ↓
Candidate Prompt v2 (score: 0.75) ✓ Better!
    ↓
GEPA Iteration 2: Further refinement
    ↓
Candidate Prompt v3 (score: 0.85) ✓ Even Better!
    ↓
Final Optimized Prompt
```

### Important Notes:
- ⏱️ **Runtime**: Depends on dataset size and max_metric_calls (5-30 minutes typical)
- 💰 **Cost**: Proportional to max_metric_calls (optimize for your budget)
- 📝 **Tracking**: All versions are tracked in MLflow for reproducibility
- 🔄 **Iterative**: Can run optimization multiple times with different data

In [ ]:
# Step 6: Run optimization (this will take a few minutes)
# NOTE: This requires valid OpenAI API key. Showing the structure:

print("Starting prompt optimization...")
print("=" * 60)
print(f"Dataset size: {len(dataset)} examples")
print(f"Target prompt: {prompt.uri}")
print(f"Optimizer: GEPA with gpt-4 reflection")
print(f"Scorers: Correctness (gpt-4-mini)")
print("=" * 60)

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=optimizer,
    scorers=scorers
)

# Access the optimized prompt
optimized_prompt = result.optimized_prompts[0]
print("\n✅ Optimization Complete!")
print(f"Original template:\n{prompt.template[:150]}...")
print(f"\nOptimized template:\n{optimized_prompt.template[:150]}...")
print(f"\nOptimized prompt URI: {optimized_prompt.uri}")

print("\n📌 To run optimization with your own data:")
print("   1. Set OPENAI_API_KEY environment variable")
print("   2. Ensure you have valid credits")
print("   3. Uncomment the mlflow.genai.optimize_prompts() call")

2025/11/21 16:53:42 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Starting prompt optimization...
Dataset size: 15 examples
Target prompt: prompts:/medical_section_classifier/1
Optimizer: GEPA with gpt-4 reflection
Scorers: Correctness (gpt-4-mini)


C:\Users\Anshu Pandey\AppData\Roaming\Python\Python310\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: Failed to determine whether UCVolumeDatasetSource can resolve source information for 'prompt_optimization_train_data'. Exception: 
  return _dataset_source_registry.resolve(
C:\Users\Anshu Pandey\AppData\Roaming\Python\Python310\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
GEPA Optimization:   5%|▌         | 15/300 [00:18<05:45,  1.21s/rollouts]

Iteration 0: Base program full valset score: 0.8
Iteration 1: Selected program 0 score: 0.8
Iteration 1: Proposed new text for medical_section_classifier: You are given a single sentence from a medical research paper. Your task is to classify the sentence into one of five predefined standard sections typically found in such papers: BACKGROUND, OBJECTIVE, METHODS, RESULTS, or CONCLUSIONS.

Definitions for each section to guide your classification:

- BACKGROUND: Sentences that provide context, rationale, or underlying scientific or clinical information to justify why the study was conducted. This includes descriptions of the problem, existing knowledge gaps, or previous findings that motivate the research.

- OBJECTIVE: Sentences that explicitly state the aim, purpose, or goal of the study. This typically involves describing what the study intends to investigate or accomplish. Objective sentences often use future or intended action language (e.g., "to evaluate," "we aim to").

- METHODS

GEPA Optimization:  12%|█▏        | 36/300 [01:01<07:47,  1.77s/rollouts]

...

Iteration 76: All subsample scores perfect. Skipping.
Iteration 76: Reflective mutation did not propose a new candidate
Iteration 77: Selected program 1 score: 1.0


GEPA Optimization:  88%|████████▊ | 264/300 [14:41<04:07,  6.86s/rollouts]

Iteration 77: All subsample scores perfect. Skipping.
Iteration 77: Reflective mutation did not propose a new candidate
Iteration 78: Selected program 1 score: 1.0


GEPA Optimization:  89%|████████▉ | 267/300 [15:02<03:50,  6.97s/rollouts]

Iteration 78: All subsample scores perfect. Skipping.
Iteration 78: Reflective mutation did not propose a new candidate
Iteration 79: Selected program 1 score: 1.0
Note: LLM call would be made here. Error: Connection error.
Note: LLM call would be made here. Error: Connection error.


## Understanding the Optimization Results

After optimization completes, you get a `PromptOptimizationResult` object with valuable information:

### Result Object Properties:

| Property | Description |
|----------|-------------|
| `optimized_prompts` | List of optimized PromptVersion objects |
| `optimizer_name` | Name of optimizer used (e.g., "GepaPromptOptimizer") |
| `initial_eval_score` | Performance score of initial prompt |
| `final_eval_score` | Performance score of optimized prompt |



In [17]:
# Step 7: Understanding and using the results

# Example of how to access result properties
def analyze_optimization_result(result):
    """
    Helper function to analyze and display optimization results
    """
    print("\n" + "=" * 70)
    print("OPTIMIZATION RESULT ANALYSIS")
    print("=" * 70)
    
    # Optimized prompts
    print(f"\n📋 Optimized Prompts: {len(result.optimized_prompts)} prompt(s)")
    for i, prompt in enumerate(result.optimized_prompts, 1):
        print(f"\n  [{i}] {prompt.name}")
        print(f"      Version: {prompt.version}")
        print(f"      URI: {prompt.uri}")
        print(f"      Template preview: {prompt.template[:100]}...")
    
    # Performance metrics
    print(f"\n📊 Performance Metrics:")
    print(f"    Optimizer: {result.optimizer_name}")
    print(f"    Initial Score: {result.initial_eval_score:.3f}")
    print(f"    Final Score:   {result.final_eval_score:.3f}")
    
    if result.initial_eval_score > 0:
        improvement_pct = ((result.final_eval_score - result.initial_eval_score) / 
                          result.initial_eval_score * 100)
        print(f"    Improvement: +{improvement_pct:.1f}%")
    
    return result

analyze_optimization_result(result)


OPTIMIZATION RESULT ANALYSIS

📋 Optimized Prompts: 1 prompt(s)

  [1] medical_section_classifier
      Version: 2
      URI: prompts:/medical_section_classifier/2
      Template preview: Classify a given sentence from a medical research paper into one of the following sections: CONCLUSI...

📊 Performance Metrics:
    Optimizer: GepaPromptOptimizer
    Initial Score: 0.800
    Final Score:   1.000
    Improvement: +25.0%


PromptOptimizationResult(optimized_prompts=[PromptVersion(name=medical_section_classifier, version=2, template=Classify a given sentence from...)], optimizer_name='GepaPromptOptimizer', initial_eval_score=0.8, final_eval_score=1.0)

## Advanced: Multi-Prompt Optimization

For complex workflows with multiple prompts, you can optimize them together. This is useful for:
- **Agent systems** with planning + execution prompts
- **Multi-turn conversations** with system + user prompts
- **Chained workflows** where outputs feed into next prompts

### When to Use Multi-Prompt Optimization:
✅ Prompts are interdependent  
✅ You have evaluation data for the full pipeline  
✅ You want joint optimization across all prompts  
✅ Performance of pipeline matters more than individual prompts  

### Example: Question Answering System with Planning
```
Plan Prompt: Generate a plan to answer the question
    ↓ (plan output fed into next prompt)
Answer Prompt: Use the plan to generate the answer
```

**Key Point**: Your predict_fn must load and use ALL prompts in the correct order!

In [ ]:
# Advanced Example: Multi-Prompt Optimization

# Register multiple related prompts
plan_prompt = mlflow.genai.register_prompt(
    name="planning_prompt",
    template="Generate a step-by-step plan to answer this question: {{question}}"
)

answer_prompt = mlflow.genai.register_prompt(
    name="answering_prompt", 
    template="Answer the question using this plan.\nQuestion: {{question}}\nPlan: {{plan}}"
)

print("Registered multi-prompt workflow:")
print(f"  1. Plan Prompt: {plan_prompt.uri}")
print(f"  2. Answer Prompt: {answer_prompt.uri}")

# Define predict_fn for multi-prompt workflow
def multi_prompt_predict_fn(question: str) -> str:
    """
    Multi-prompt prediction function:
    1. Load and use plan prompt
    2. Get plan from LLM
    3. Load and use answer prompt with the plan
    4. Return final answer
    """
    # Load plan prompt
    plan_template = mlflow.genai.load_prompt("prompts:/planning_prompt@latest")
    plan_formatted = plan_template.format(question=question)
    
    # Get plan (simulated)
    plan = "Step 1: ... Step 2: ... Step 3: ..."  # In reality, call LLM
    
    # Load answer prompt
    answer_template = mlflow.genai.load_prompt("prompts:/answering_prompt@latest")
    answer_formatted = answer_template.format(question=question, plan=plan)
    
    # Get final answer (simulated)
    final_answer = "The answer is..."  # In reality, call LLM
    
    return final_answer

print("\nMulti-prompt predict function defined")
print("\nTo optimize multiple prompts together:")
print("""
result = mlflow.genai.optimize_prompts(
    predict_fn=multi_prompt_predict_fn,
    train_data=dataset,
    prompt_uris=[
        plan_prompt.uri,
        answer_prompt.uri
    ],
    optimizer=optimizer,
    scorers=scorers
)
""")

### Accessing Results:

### Before and After Example:
**Before Optimization:**
```
Classify this medical research paper sentence into one of these 
sections: CONCLUSIONS, RESULTS, METHODS, OBJECTIVE, BACKGROUND.

Sentence: {{sentence}}
```

**After Optimization (typical):**
```
You are an expert medical research paper classifier. Your task is to 
categorize single sentences into abstract sections.

Classification Categories:
- BACKGROUND: Context, prior knowledge, unmet need
- OBJECTIVE: Study aims, purposes, hypotheses
- METHODS: Study design, procedures, measurements
- RESULTS: Observed findings, quantified outcomes
- CONCLUSIONS: Interpretations, implications, recommendations

Carefully analyze the sentence and return exactly one category label.
Input: {{sentence}}
Output: [category label only]
```

## Advanced: Custom Scorers and Aggregation

While built-in scorers are convenient, custom scorers give you fine-grained control over what "better" means for your use case.

### Creating Custom Scorers:
```python
@scorer
def my_scorer(outputs: Any, expectations: dict) -> float:
    # Your custom scoring logic
    # Return a score between 0.0 and 1.0
    return score
```

### Aggregating Multiple Scores:
When you have multiple scorers, you can combine them with custom logic:

```python
def weighted_objective(scores: dict) -> float:
    # Combine different metrics with weights
    return 0.6 * scores['correctness'] + 0.4 * scores['brevity']
```

### Why Custom Scorers?
- 🎯 **Domain-specific metrics**: Medical accuracy, legal compliance, etc.
- ⚖️ **Weighted objectives**: Optimize for multiple criteria simultaneously
- 📏 **Soft metrics**: Stylistic preferences, tone, formality
- 🔐 **Constraints**: Ensure outputs meet minimum requirements

In [ ]:
# Advanced Example: Custom Scorers with Weighted Aggregation

from mlflow.genai.scorers import scorer

# Define domain-specific custom scorers

@scorer
def exact_match_scorer(outputs: Any, expectations: dict) -> float:
    """
    Exact match between output and expected classification.
    """
    output_upper = str(outputs).strip().upper()
    expected_upper = str(expectations.get("expected_response", "")).strip().upper()
    return 1.0 if output_upper == expected_upper else 0.0

@scorer
def category_validity_scorer(outputs: Any, expectations: dict) -> float:
    """
    Ensure output is from the valid set of categories.
    Partial credit for containing valid keywords.
    """
    valid_categories = {"CONCLUSIONS", "RESULTS", "METHODS", "OBJECTIVE", "BACKGROUND"}
    output_str = str(outputs).strip().upper()
    
    # Perfect score for exact category
    if output_str in valid_categories:
        return 1.0
    
    # Partial score if output contains valid category name
    for cat in valid_categories:
        if cat in output_str:
            return 0.6
    
    # No score for invalid output
    return 0.0

@scorer
def conciseness_scorer(outputs: Any, expectations: dict) -> float:
    """
    Prefer shorter, concise outputs (no explanation, just the label).
    Max 30 characters.
    """
    output_length = len(str(outputs))
    max_length = 30
    return max(0.0, 1.0 - (output_length - max_length) / max_length)

# Define aggregation function
def weighted_objective(scores: dict) -> float:
    """
    Combine multiple metrics:
    - 70% weight on exact match (most important)
    - 20% weight on validity (never output invalid categories)
    - 10% weight on conciseness (prefer single-word outputs)
    """
    return (
        0.7 * scores.get('exact_match_scorer', 0.0) +
        0.2 * scores.get('category_validity_scorer', 0.0) +
        0.1 * scores.get('conciseness_scorer', 0.0)
    )

print("Custom scorers defined:")
print("  1. exact_match_scorer: Exact classification accuracy")
print("  2. category_validity_scorer: Output must be valid category")
print("  3. conciseness_scorer: Prefer single-word outputs")
print("\nWeighted aggregation: 70% accuracy + 20% validity + 10% conciseness")

# Example: Using custom scorers in optimization
custom_scorers = [
    exact_match_scorer,
    category_validity_scorer,
    conciseness_scorer
]

print("\nTo use custom scorers in optimization:")
print("""
result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=optimizer,
    scorers=custom_scorers,
    aggregation=weighted_objective  # Combine multiple scores
)
""")

## Integration with LangChain

MLflow prompt optimization works seamlessly with LangChain. The key difference is using `to_single_brace_format()` instead of `format()`.

### Why the Difference?
- **MLflow Prompt Registry** uses double braces: `{{variable}}`
- **LangChain PromptTemplate** uses single braces: `{variable}`
- Use `to_single_brace_format()` to convert between formats

### LangChain Integration Example:
```python
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.llms import OpenAI

def predict_fn(question: str) -> str:
    # Load from MLflow registry
    mlflow_prompt = mlflow.genai.load_prompt("prompts:/qa@latest")
    
    # Convert to single-brace format for LangChain
    template_str = mlflow_prompt.to_single_brace_format()
    
    # Create LangChain components
    prompt = PromptTemplate(input_variables=["question"], template=template_str)
    llm = OpenAI()
    chain = LLMChain(llm=llm, prompt=prompt)
    
    # Run chain
    result = chain.run(question=question)
    return result
```

### Frameworks Supported:
✅ LangChain  
✅ OpenAI Agent  
✅ CrewAI  
✅ DSPy  
✅ AutoGen  
✅ Custom implementations

In [ ]:
# Integration with LangChain Example

def langchain_example():
    """
    Shows how to integrate MLflow prompt optimization with LangChain.
    """
    
    langchain_code = '''
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.llms import OpenAI
import mlflow

# Step 1: Define predict function for LangChain workflow
def predict_fn_langchain(question: str) -> str:
    """
    Prediction function using LangChain with MLflow-optimized prompts
    """
    # Load prompt from MLflow registry
    mlflow_prompt = mlflow.genai.load_prompt("prompts:/qa@latest")
    
    # Convert double-brace format to single-brace for LangChain
    template_str = mlflow_prompt.to_single_brace_format()
    
    # Create LangChain prompt template
    prompt_template = PromptTemplate(
        input_variables=["question"],
        template=template_str
    )
    
    # Create LLM and chain
    llm = OpenAI(temperature=0.7)
    chain = LLMChain(llm=llm, prompt=prompt_template)
    
    # Run the chain
    result = chain.run(question=question)
    return result

# Step 2: Optimize using the same predict_fn
result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn_langchain,
    train_data=dataset,
    prompt_uris=["prompts:/qa@latest"],
    optimizer=optimizer,
    scorers=scorers
)

# Step 3: Use optimized prompt in production
optimized = result.optimized_prompts[0]
print(f"Optimized URI: {optimized.uri}")
    '''
    
    print("LangChain Integration Example:")
    print("=" * 70)
    print(langchain_code)

langchain_example()

## Common Use Cases and Solutions

### Use Case 1: Improving Accuracy
**Problem**: Your prompts are producing incorrect outputs  
**Solution**: Use Correctness scorer focused on exact match

```python
from mlflow.genai.scorers import Correctness

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="openai:/gpt-4"),
    scorers=[Correctness(model="openai:/gpt-4")]
)
```

### Use Case 2: Optimizing for Safety/Compliance
**Problem**: Need to ensure outputs are safe or compliant  
**Solution**: Use Safety scorer or custom domain validators

```python
from mlflow.genai.scorers import Safety

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=GepaPromptOptimizer(reflection_model="openai:/gpt-4"),
    scorers=[Safety(model="openai:/gpt-4")]
)
```

### Use Case 3: Model Migration (Reduce Costs)
**Problem**: Need to switch from expensive model (gpt-4) to cheaper (gpt-4-mini)  
**Solution**: Optimize prompts using your production data

```python
# Use your production history as training data
production_data = [
    {
        "inputs": {"question": input_text},
        "expectations": {"expected_response": actual_output}
    }
    for input_text, actual_output in production_history
]

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=production_data,
    prompt_uris=[prompt.uri],
    optimizer=GepaPromptOptimizer(
        reflection_model="openai:/gpt-4",  # Powerful model for optimization
        max_metric_calls=500  # Larger budget for important migration
    ),
    scorers=[Correctness(model="openai:/gpt-4-mini")]  # Evaluate with target model
)
```

### Use Case 4: Multi-Objective Optimization
**Problem**: Need to balance accuracy, cost, latency, and safety  
**Solution**: Use multiple scorers with weighted aggregation

```python
def multi_objective(scores: dict) -> float:
    return (
        0.5 * scores['correctness'] +      # 50% accuracy
        0.2 * scores['safety'] +            # 20% safety
        0.15 * scores['fluency'] +          # 15% fluency
        0.15 * scores['conciseness']        # 15% conciseness
    )

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=dataset,
    prompt_uris=[prompt.uri],
    optimizer=optimizer,
    scorers=[
        Correctness(model="openai:/gpt-4"),
        Safety(model="openai:/gpt-4"),
        fluency_scorer,
        conciseness_scorer
    ],
    aggregation=multi_objective
)
```

## Troubleshooting Common Issues

### ❌ Issue: Optimization Takes Too Long

**Symptoms**: Process running for hours without completion

**Causes & Solutions**:
```python
# 1. Reduce dataset size for faster iteration
small_dataset = dataset[:10]  # Start with small subset for testing

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=small_dataset,  # Smaller dataset = faster iteration
    ...
)

# 2. Reduce max_metric_calls budget
optimizer = GepaPromptOptimizer(
    reflection_model="openai:/gpt-4",
    max_metric_calls=100  # Reduce from default
)

# 3. Use faster reflection model (less powerful but faster)
optimizer = GepaPromptOptimizer(
    reflection_model="openai:/gpt-4-mini",  # Faster than gpt-4
    max_metric_calls=100
)
```

### ❌ Issue: No Improvement Observed

**Symptoms**: Final score same or worse than initial score

**Causes & Solutions**:
```python
# 1. Verify scorer accuracy
# Check if your Correctness scorer actually measures what you care about
def debug_scorer(outputs: Any, expectations: dict) -> float:
    print(f"Output: {outputs}")
    print(f"Expected: {expectations}")
    return 1.0 if outputs == expectations.get("expected_response") else 0.0

# 2. Increase dataset diversity
# Add more varied examples covering edge cases
dataset.extend([
    {"inputs": {"sentence": edge_case}, "expectations": {...}}
    for edge_case in hard_cases
])

# 3. Improve expectations quality
# Verify all ground truth labels are correct
for example in dataset:
    assert is_valid_label(example['expectations']['expected_response'])
```

### ❌ Issue: "Prompts Not Being Used" Error

**Symptoms**: Optimization runs but optimized prompt is unchanged

**Causes**: predict_fn not loading from registry

```python
# ✅ CORRECT - Loads from registry each time
def predict_fn(sentence: str) -> str:
    prompt = mlflow.genai.load_prompt("prompts:/medical_classifier@latest")
    return llm_call(prompt.format(sentence=sentence))

# ❌ WRONG - Hardcoded prompt, won't be optimized
def predict_fn(sentence: str) -> str:
    hardcoded = "Classify: {{sentence}}"  # ← Problem!
    return llm_call(hardcoded.format(sentence=sentence))

# ❌ WRONG - Prompt loaded once outside function
prompt = mlflow.genai.load_prompt("prompts:/medical_classifier@latest")
def predict_fn(sentence: str) -> str:  # ← Won't load new versions
    return llm_call(prompt.format(sentence=sentence))
```

### ❌ Issue: API Rate Limits or Quota Exceeded

**Symptoms**: Errors about exceeding API rate limits

**Solutions**:
```python
# 1. Reduce max_metric_calls
optimizer = GepaPromptOptimizer(
    reflection_model="openai:/gpt-4",
    max_metric_calls=50  # Much lower
)

# 2. Use smaller dataset
small_dataset = dataset[:5]

# 3. Increase time between runs
import time
time.sleep(60)  # Wait before next optimization run

# 4. Use cheaper models
optimizer = GepaPromptOptimizer(
    reflection_model="openai:/gpt-4-mini",
    max_metric_calls=100
)
```

### ❌ Issue: Format Mismatch Error

**Symptoms**: "expectations format does not match outputs format"

**Solutions**:
```python
# Ensure expectations format matches predict_fn return type

# If predict_fn returns string:
dataset = [
    {
        "inputs": {"text": "..."},
        "expectations": {"expected_response": "LABEL"}  # string, not dict
    }
]

# If predict_fn returns dict:
dataset = [
    {
        "inputs": {"text": "..."},
        "expectations": {"response": {"classification": "LABEL"}}  # dict format
    }
]
```

## Best Practices for Production

### 1. Version Control Your Optimized Prompts
```python
# Register optimized prompts with meaningful names
optimized_prompt = result.optimized_prompts[0]

mlflow.genai.register_prompt(
    name="medical_classifier_optimized_nov2024",
    template=optimized_prompt.template
)
```

### 2. A/B Test Before Full Rollout
```python
# Compare original vs optimized on holdout test set
original_score = evaluate_prompt(original_prompt, test_data)
optimized_score = evaluate_prompt(optimized_prompt, test_data)

if optimized_score > original_score * 1.05:  # 5% improvement threshold
    deploy_to_production(optimized_prompt)
else:
    print("Improvement not significant, keeping original")
```

### 3. Continuously Monitor Performance
```python
# Track prompt performance in production
def log_prediction(prompt_uri, input_data, output, ground_truth=None):
    """Log predictions for monitoring and future optimization"""
    mlflow.log_metric("prompt_accuracy", 1.0 if output == ground_truth else 0.0)
    mlflow.log_param("prompt_uri", prompt_uri)
```

### 4. Regular Re-optimization
```python
# Re-optimize monthly with new data
production_data_this_month = collect_labeled_examples()

result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=production_data_this_month,
    prompt_uris=[current_prompt.uri],
    optimizer=optimizer,
    scorers=scorers
)
```

### 5. Document Optimization Decisions
```python
# Record why you optimized and what changed
metadata = {
    "optimization_date": "2024-11-21",
    "reason": "Accuracy dropped below 85% threshold",
    "dataset_size": len(dataset),
    "initial_score": result.initial_eval_score,
    "final_score": result.final_eval_score,
    "scorer": "Correctness",
    "optimizer": "GEPA"
}

# Store with optimized prompt
mlflow.log_dict(metadata, "optimization_metadata.json")
```

### 6. Keep Original Prompts for Rollback
```python
# Never delete original prompts
# Always able to revert if optimized version underperforms

original_uri = "prompts:/medical_classifier/1"
optimized_uri = "prompts:/medical_classifier/2"

# If needed
if production_issues_detected(optimized_uri):
    switch_to_prompt(original_uri)  # Quick rollback
```

## Key Takeaways

### 🎯 Core Concepts
1. **Prompt optimization** = systematic data-driven improvement of prompts
2. **GEPA algorithm** = LLM-driven reflection + iterative refinement
3. **Components needed** = prompt + prediction function + dataset + optimizer + scorers
4. **No lock-in** = works with any framework (LangChain, OpenAI Agent, etc.)

### 🔄 Optimization Workflow
1. Register initial prompt → 2. Define predict function → 3. Prepare dataset
4. Configure optimizer → 5. Create scorers → 6. Run optimization
7. Evaluate results → 8. Deploy to production → 9. Monitor performance

### 💡 Pro Tips
- Start with small datasets to iterate quickly
- Use expensive models (gpt-4) for reflection, cheaper for scoring
- Combine multiple scorers with weighted aggregation for complex objectives
- Always A/B test before production deployment
- Keep original prompts for quick rollback
- Document your optimization decisions for future reference
- Re-optimize monthly with new production data

### 📚 Resources
- MLflow Docs: https://mlflow.org/docs/latest/genai/prompt-registry/optimize-prompts/
- GEPA Paper: https://arxiv.org/abs/2507.19457
- MLflow Prompt Registry: https://mlflow.org/docs/latest/genai/prompt-registry/

### 🚀 Next Steps
1. Set up your OpenAI API credentials
2. Prepare your training dataset (20-30+ examples)
3. Define your evaluation metrics
4. Run optimization on a small subset first
5. Evaluate on production data
6. Deploy to production gradually